In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")

import re

import pandas as pd
from datasets import disable_caching, load_from_disk
from dotenv import load_dotenv
from hydra import compose, initialize

from hallucinations_kg.defaults import ROOT_PATH

disable_caching()

## Loading

In [3]:
load_dotenv()

True

In [4]:
with initialize(version_base="1.3", config_path=str("config")):
    cfg = compose(config_name="evaluate_correction")

In [5]:
dataset = load_from_disk(ROOT_PATH / "data/correction/evaluate/wiki_bio/Llama-3.1-70B-Instruct")
dataset = dataset["evaluation"]
df = dataset.to_pandas()

In [6]:
def parse_answer(answer: str) -> str:
    answer = re.findall(r"[\w]+|[.,!?;\"']", answer)
    if "yes" in answer:
        return "factual"
    elif "no" in answer:
        return "non-factual"
    elif "refused" in answer:
        return "refused"
    else:
        raise ValueError(f"Unknown answer: {answer.__repr__()}")

In [7]:
sentence_df = df.explode(list(df.columns))
sentence_df = sentence_df.map(parse_answer)

In [11]:
results = []


def get_method_name(col):
    if "baseline" in col:
        return "baseline"
    elif "facts" in col:
        return "fact"
    elif "sentence" in col:
        return "sentence"
    else:
        raise ValueError(f"Unknown method: {col}")


for col in sentence_df.columns:
    print(col)
    print(sentence_df[col].value_counts().to_dict())
    print("\n")
    results.append(
        {"method": get_method_name(col), **sentence_df[col].value_counts(normalize=True).to_dict()}
    )


results = pd.DataFrame(results)
results = results[["method", "factual", "non-factual", "refused"]]
baseline_results = results[results["method"] == "baseline"].iloc[0]


def format_values(x, baseline_value):
    diff = (x - baseline_value) / baseline_value * 100
    if diff == 0:
        return f"{x:.2f}"
    plus = "+" if diff > 0 else ""
    return f"{x:.2f} ({plus}{diff:.1f}%)"


for col in results.columns:
    if col != "method":
        results[col] = results[col].apply(format_values, args=(baseline_results[col],))


results.sort_values(by="factual", ascending=False)

correction_baseline_corrected_sentences_evaluations
{'non-factual': 1404, 'factual': 434, 'refused': 70}


correction_sentences_corrected_sentences_evaluations
{'non-factual': 1337, 'factual': 480, 'refused': 91}


correction_facts_corrected_sentences_evaluations
{'non-factual': 1229, 'factual': 588, 'refused': 91}




,method,factual,non-factual,refused
2,fact,0.31 (+35.5%),0.64 (-12.5%),0.05 (+30.0%)
1,sentence,0.25 (+10.6%),0.70 (-4.8%),0.05 (+30.0%)
0,baseline,0.23,0.74,0.04


In [12]:
print(results.to_latex(index=False))

\begin{tabular}{llll}
\toprule
method & factual & non-factual & refused \\
\midrule
baseline & 0.23 & 0.74 & 0.04 \\
sentence & 0.25 (+10.6%) & 0.70 (-4.8%) & 0.05 (+30.0%) \\
fact & 0.31 (+35.5%) & 0.64 (-12.5%) & 0.05 (+30.0%) \\
\bottomrule
\end{tabular}

